## Questão 8 - Sistema de recomendação

### Cenário

A Marina percebeu que clientes que compram lanchas quase sempre esquecem de levar a
defensa (proteção lateral). Ela quer implementar uma vitrine de "Quem comprou isso, também
levou…" no site. 

Como não temos ferramentas de Big Data caras, você precisará criar um motor de recomendação, baseado na similaridade de compra dos clientes. Identificar qual produto deve ser recomendado junto ao item "GPS Garmin Vortex Mare Drift", com base na similaridade de comportamento de compra dos clientes.

### Tarefa:

1. Crie uma matriz de interação Usuário × Produto obedecendo às regras abaixo:
    * Linhas: `id_cliente`
    * Colunas: `id_produto`
    * Valor da célula:
        * 1 se o cliente comprou ao menos uma vez o produto
        * 0 caso contrário
    *  ignore a quantidade comprada (presença/ausência apenas)
2. Cálculo de Similaridade entre Produto
    * Calcule a Similaridade de Cosseno (Cosine Similarity) entre os vetores dos produtos
    * A similaridade deve ser calculada `produto * produto`, com base nos clientes que compraram cada item
3. Ranking de Produtos Similares
    * Considere o produto "GPS Garmin Vortex Mare Drift" como item de referencia
    * Gere um ranking com os 5 produtos mais similares a ele
    * Desconsidere o próprio GPS no ranking

In [18]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
# Configurações de exibição
pd.set_option('display.max_columns', None)

In [6]:
# Carregando o dataset processado
df = pd.read_csv('../datasets/processed/vendas_cambio_custos.csv')

PRODUTO = "GPS Garmin Vortex Maré Drift"
df.head()

,id,id_client,id_product,qtd,total,sale_date,data_cambio,cotacao_dolar,start_date,usd_price,product_name
0,1230,17,91,4,512566.80,2023-01-01,2023-01-01,5.2177,2022-03-16,26303.31,Motor Elétrico Tohatsu Zenith Oceanic 113HP
1,8211,14,67,3,257367.00,2023-01-01,2023-01-01,5.2177,2022-06-17,16720.73,Motor de Popa Yamaha Mako 108HP
2,666,14,15,5,132524.05,2023-01-01,2023-01-01,5.2177,2022-12-06,5325.40,Radar Garmin Tidal Thrust
3,5256,19,78,12,1461139.00,2023-01-01,2023-01-01,5.2177,2022-10-14,23053.04,Motor Elétrico Honda Mako Axis 131HP
4,3549,10,53,13,662886.25,2023-01-01,2023-01-01,5.2177,2022-07-19,9958.63,Motor Elétrico Torqeedo Zen Titan Hydra 129HP


> 1. Crie uma matriz de interação Usuário × Produto 

In [ ]:
def gerar_matriz_binaria(df):
    # Pivotagem por contagem para identificar interações
    pivot = df.pivot_table(index='id_client', columns='id_product', values='total', aggfunc='count', fill_value=0)
    
    # 1 se comprou, 0 caso contrário (independente da quantidade)
    return pd.DataFrame(np.where(pivot.values > 0, 1, 0), index=pivot.index, columns=pivot.columns)

matriz_binaria = gerar_matriz_binaria(df)
print(f"Matriz de Interação pronta: {matriz_binaria.shape[0]} usuários e {matriz_binaria.shape[1]} produtos.")
matriz_binaria.head()

Matriz de Interação pronta: 49 usuários e 150 produtos.


id_product,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150
id_client,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,1,0,1,1,1,1,0,0,0,0,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,0,1,1,1,1,1,0,1,1,1,1,0,1,0,1,0,1,1,1,1,0,1,1,1,0,1,1,0,1,1,1,1,1,0,1,0,1,1,1,1,1,1,0,0,0,1,1,1,1,0,1,1,1,0,1,1,0,1,1,1,0,1,0,1,1,0,1,0,1,1,0,0,1,1,1,1,1,0,1,0,1,0,1,1,1,1,1,1,1,1,1,1,0,1,0,1,0,0,0,0,1,1,0,1,1,1,1,1,1,0,1,1,1,1,1,0,0
2,0,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,1,1,0,0,1,1,0,1,1,0,1,0,1,1,0,0,1,1,0,1,1,1,1,1,1,0,0,1,0,1,0,0,1,1,0,1,1,1,1,0,0,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,0,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,1
3,1,1,1,1,1,1,1,0,1,1,1,1,1,0,1,0,0,1,0,0,1,0,1,1,1,1,0,1,1,1,1,1,1,1,0,0,1,1,0,1,0,1,1,1,1,1,1,1,1,0,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,0,1,1,0,0,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,0,1,1,1,1,0,1,0,0,1,1,1,1,1,1,1,1,1,0,1,0,1,1,1,0,1,1,1,0,0,1,1,1,1,1,1,1,1,0,1,0,1,0,1,0,1,1,1,0,1,1,1,1,1
4,1,1,0,1,1,0,1,0,0,1,1,1,1,1,1,0,0,0,1,1,0,1,1,0,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,1,0,1,0,1,0,1,1,1,1,0,0,1,0,1,1,0,1,1,1,1,1,1,0,1,0,1,1,1,1,1,1,1,0,0,1,0,0,0,1,1,1,0,1,1,0,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,0,1,1,1,1,1,0,1,1,0,1,0,0,1,1,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,0,0,1,0,1,1
5,1,1,0,1,1,1,1,1,1,1,1,1,0,1,0,0,0,1,1,1,0,1,0,0,1,1,0,0,1,0,1,1,1,0,1,0,1,0,1,1,1,1,1,1,1,1,1,1,1,1,0,1,1,0,1,1,0,1,0,1,1,1,1,0,0,0,0,1,1,1,1,1,1,1,0,1,0,1,1,1,1,0,1,1,1,1,1,1,1,1,0,0,1,0,1,1,0,0,1,1,1,0,1,1,1,0,1,1,1,0,1,1,1,1,1,0,1,1,1,1,1,1,1,1,0,0,0,1,1,1,1,1,1,0,0,1,0,1,1,1,1,1,1,1,0,1,1,1,1,0


In [21]:
# Transposição para cálculo Produto x Produto
similaridade_matrix = cosine_similarity(matriz_binaria.T)

# Conversão para DataFrame indexado
df_sim = pd.DataFrame(similaridade_matrix, index=matriz_binaria.columns, columns=matriz_binaria.columns)

df_sim.head()

id_product,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150
id_product,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.775058,0.737865,0.810191,0.748331,0.769484,0.775058,0.727825,0.711512,0.698771,0.892149,0.764217,0.759257,0.748331,0.846432,0.801784,0.764217,0.795133,0.801784,0.737865,0.790569,0.790569,0.737865,0.715626,0.814877,0.814877,0.850000,0.775000,0.795133,0.764217,0.769484,0.727825,0.775000,0.805807,0.829515,0.754673,0.843274,0.709952,0.790569,0.721605,0.759257,0.825723,0.675303,0.860828,0.775058,0.790569,0.759257,0.769484,0.786373,0.769484,0.819813,0.795133,0.721688,0.698771,0.748331,0.753819,0.759257,0.764217,0.753819,0.743151,0.860828,0.795133,0.764217,0.754673,0.810191,0.764217,0.869626,0.743151,0.790569,0.764217,0.816922,0.825000,0.779813,0.814877,0.675303,0.779813,0.769484,0.759257,0.790569,0.795133,0.843274,0.748331,0.704664,0.753819,0.805807,0.705024,0.748331,0.709952,0.779813,0.721605,0.759257,0.732140,0.786373,0.764217,0.681554,0.748331,0.653156,0.839570,0.748331,0.726722,0.753819,0.732140,0.819813,0.748331,0.743151,0.748331,0.790569,0.790569,0.727825,0.786373,0.843274,0.835510,0.786373,0.790569,0.825000,0.727825,0.681554,0.810191,0.825000,0.770675,0.715626,0.892149,0.770675,0.753819,0.737865,0.775058,0.805807,0.810191,0.711512,0.769484,0.805807,0.759257,0.748331,0.737865,0.743834,0.711512,0.727825,0.800000,0.825000,0.705024,0.795133,0.753819,0.775058,0.769484,0.721605,0.775058,0.872082,0.721605,0.727825,0.775058
2,0.775058,1.000000,0.704295,0.757865,0.714286,0.712931,0.771429,0.750290,0.788811,0.687256,0.799086,0.676123,0.666737,0.742857,0.712931,0.685714,0.704295,0.712931,0.714286,0.676123,0.676123,0.732467,0.732467,0.676763,0.844742,0.844742,0.748331,0.775058,0.767772,0.788811,0.685511,0.694713,0.721605,0.805867,0.808543,0.747018,0.816982,0.667894,0.676123,0.800000,0.724714,0.676763,0.659153,0.757865,0.714286,0.732467,0.666737,0.712931,0.695725,0.767772,0.773309,0.712931,0.648074,0.597614,0.685714,0.694713,0.695725,0.788811,0.694713,0.706188,0.730798,0.767772,0.704295,0.627495,0.703732,0.760639,0.732467,0.765037,0.788811,0.760639,0.704295,0.721605,0.722501,0.765547,0.533600,0.778078,0.767772,0.666737,0.732467,0.822613,0.760639,0.714286,0.690541,0.722501,0.750290,0.637748,0.714286,0.667894,0.722501,0.800000,0.753702,0.637748,0.666737,0.732467,0.758971,0.685714,0.667894,0.765547,0.742857,0.687256,0.722501,0.724714,0.824863,0.714286,0.735612,0.685714,0.760639,0.760639,0.750290,0.753702,0.732467,0.784931,0.724714,0.788811,0.801784,0.778078,0.637536,0.784931,0.775058,0.647339,0.647339,0.824863,0.706188,0.722501,0.704295,0.800000,0.778078,0.811998,0.760639,0.795192,0.750290,0.753702,0.771429,0.647952,0.685511,0.704295,0.750290,0.775058,0.775058,0.724714,0.795192,0.778078,0.771429,0.767772,0.685714,0.742857,0.712931,0.628571,0.750290,0.742857
3,0.737865,0.704295,1.000000,0.800641,0.704295,0.865181,0.732467,0.712396,0.777778,0.707107,0.813326,0.722222,0.743161,0.619780,0.811107,0.760639,0.694444,0.757033,0.704295,0.750000,0.777778,0.777778,0.750000,0.783349,0.806898,0.806898,0.764217,0.869626,0.757033,0.777778,0.784070,0.712396,0.737865,0.739795,0.797234,0.648181,0.750000,0.628619,0.722222,0.704295,0.685994,0.725324,0.618984,0.747265,0.732467,0.694444,0.685994,0.757033,0.657411,0.729996,0.838742,0.838144,0.760726,0.707107,0.760639,0.767195,0.771744,0.750000,0.739795,0.725324,0.747265,0.729996,0.750000,0.707107,0.800641,0.777778,0.666667,0.667298,0.750000,0.666667,

In [17]:
def obter_recomendacoes(df_original, df_similaridade, nome_produto_alvo, top_n=5):
    # Obtendo ID do alvo
    id_alvo = df_original[df_original['product_name'] == nome_produto_alvo]['id_product'].values[0]
    
    # Gerando ranking excluindo o próprio produto
    ranking = df_similaridade[id_alvo].sort_values(ascending=False).drop(labels=[id_alvo])
    
    # Cruzando IDs com nomes para o relatório final
    resumo = []
    for id_p, score in ranking.head(top_n).items():
        nome = df_original[df_original['id_product'] == id_p]['product_name'].unique()[0]
        resumo.append({"ID": id_p, "Produto": nome, "Score": round(score, 4)})
    
    return pd.DataFrame(resumo)

df_ranking_final = obter_recomendacoes(df, df_sim, PRODUTO)
df_ranking_final

,ID,Produto,Score
0,94,Motor de Popa Volvo Magnum 276HP,0.8696
1,11,GPS Furuno Swift Leviathan Poseidon,0.8680
2,35,Radar Furuno Swift,0.8539
3,115,Cabo de Nylon Delta Force Magnum Leviathan,0.8500
4,1,Transponder AIS Maré Magnum,0.8500


In [20]:
fig = px.bar(df_ranking_final, 
             x='Score', 
             y='Produto', 
             orientation='h',
             title=f"Produtos sugeridos para venda junto com:<br><sup>{PRODUTO}</sup>",
             text='Score',
             color='Score',
             color_continuous_scale='Blues')

fig.update_layout(yaxis={'categoryorder':'total ascending'}, template='simple_white')
fig.show()